In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
import json

# Добавляем корень проекта в пути, чтобы импорты src работали корректно
project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from src.llm_platform.data_foundry.evol_pipeline import EvolPipeline

In [ ]:
input_dataset = project_root / "data/processed/test_sft_dataset.jsonl"
output_dataset = project_root / "data/processed/test_evolved_dataset.jsonl"
prompts_file = project_root / "src/llm_platform/data_foundry/evol_prompts.yaml"

# Инициализируем пайплайн
pipeline = EvolPipeline(
    input_file=input_dataset,
    output_file=output_dataset,
    prompts_file=prompts_file,
    model_name="openrouter/free",
    max_concurrent_requests=1 # Можно увеличить до 5, если API стабилен
)

# Запускаем эволюцию
await pipeline.run_evolution()

In [ ]:
with open(output_dataset, "r", encoding="utf-8") as f:
    first_line = f.readline()
    if first_line:
        evolved_pair = json.loads(first_line)
        
        print(f"ID новой пары: {evolved_pair.get('pair_id')}")
        print(f"Связанный чанк текста: {evolved_pair.get('source_chunk_id')}")
        print(f"Флаг эволюции: {evolved_pair.get('is_evolved')}\n")
        print("-" * 50)
        
        for msg in evolved_pair.get("messages", []):
            role = msg.get("role").upper()
            content = msg.get("content")
            print(f"[{role}]:\n{content}\n")
            print("-" * 50)
    else:
        print("Файл пуст. Проверь логи генерации.")